# 市场收益与市场统计指标计算

In [1]:
import numpy as np
import pandas as pd
from scipy import stats
from WindPy import w
import datetime

### Wind提供了包括全部A股的市场指数数据，我们直接使用其进行市场指标的计算

In [2]:
## 启动Wind终端的API
## 判断WindPy是否已经登录成功
w.start() 
w.isconnected()

Welcome to use Wind Quant API for Python (WindPy)!

COPYRIGHT (C) 2020 WIND INFORMATION CO., LTD. ALL RIGHTS RESERVED.
IN NO CIRCUMSTANCE SHALL WIND BE RESPONSIBLE FOR ANY DAMAGES OR LOSSES CAUSED BY USING WIND QUANT API FOR Python.


True

In [3]:
## 获得Wind A股的指数数据
mkt_trdt = w.wsd("881001.WI", "pct_chg", "2007-01-01", "2019-01-31", "PriceAdj=F")
mkt_trdt = pd.DataFrame(mkt_trdt.Data, index = mkt_trdt.Codes, columns = mkt_trdt.Times)
mkt_trdt = mkt_trdt.T
mkt_trdt = mkt_trdt.reset_index()

In [4]:
## 修改列名
## 提取日期中的年份、月份
mkt_trdt.columns = ['date', 'pct_chg']
mkt_trdt['pct_chg']

0       0.810832
1       0.900898
2       3.230241
3       2.804548
4       2.502602
          ...   
2936    0.129340
2937   -0.320885
2938   -0.596894
2939   -1.040220
2940   -0.113312
Name: pct_chg, Length: 2941, dtype: float64

In [5]:
mkt_trdt['pct_chg'] = mkt_trdt['pct_chg'] * 0.01
mkt_trdt['date'] = pd.to_datetime(mkt_trdt['date'])
mkt_trdt['year'] = mkt_trdt['date'].dt.year
mkt_trdt['month'] = mkt_trdt['date'].dt.month

In [6]:
## 读取已保存的shibor数据
shibor = w.wsd("SHIBORON.IR", "open", "2007-01-01", "2019-01-31")
shibor = pd.DataFrame(shibor.Data,index = shibor.Codes,columns = shibor.Times)
shibor = shibor.T
shibor = shibor.reset_index()
shibor.columns = ['date', 'shibor']
shibor['shibor'] = shibor['shibor'] * 0.01
shibor['date'] = pd.to_datetime(shibor['date'])

In [7]:
## 合并A股收益数据和shibor收益数据
mkt_fore = pd.merge(mkt_trdt, shibor, on = 'date', how = 'left')
mkt_fore['exr'] = mkt_fore['pct_chg'] - mkt_fore['shibor']
mkt_fore = mkt_fore[['year', 'month', 'date', 'pct_chg', 'shibor', 'exr']]

## 计算数学方差、偏度、峰度

In [8]:
## 分组，计算各月均值、方差、偏度、峰度
mkt_sta = mkt_fore.groupby([mkt_fore['year'], mkt_fore['month']])['exr'].agg([('num', 'count'), ('mean', 'mean'), ('math_var', 'var'), ('math_skew', 'skew'),('math_kurt', stats.kurtosis),])
mkt_sta = mkt_sta.reset_index()

## 计算调整后的方差、偏度、峰度

In [9]:
## 删除首行
mkt_lf = mkt_fore.groupby([mkt_fore['year'], mkt_fore['month']])['date'].first()
mkt_lf = mkt_lf.reset_index()
mkt_lf.insert(2, 'flag', np.ones(len(mkt_lf)))
mkt_lf = mkt_lf[['date', 'flag']]

## 保留其他数据
mkt_wof = pd.merge(mkt_fore, mkt_lf, on = 'date', how = 'left')
mkt_wof = mkt_wof[mkt_wof['flag'] != 1]
mkt_wof = mkt_wof[['date', 'exr']]
mkt_wof = mkt_wof.rename(columns={'exr': 'exr_f'})
mkt_wof.insert(2, 'order', np.arange(len(mkt_wof)))

In [10]:
## 删除末行
mkt_ll = mkt_fore.groupby([mkt_fore['year'], mkt_fore['month']])['date'].last()
mkt_ll = mkt_ll.reset_index()
mkt_ll.insert(2, 'flag', np.ones(len(mkt_ll)))
mkt_ll = mkt_ll[['date', 'flag']]

## 保留其他数据
mkt_wol = pd.merge(mkt_fore, mkt_ll, on = 'date', how = 'left')
mkt_wol = mkt_wol[mkt_wol['flag'] != 1]
mkt_wol = mkt_wol[['year', 'month', 'exr']]
mkt_wol = mkt_wol.rename(columns={'exr': 'exr_l'})
mkt_wol.insert(2, 'order', np.arange(len(mkt_wol)))

In [11]:
## 合并去除第一行/最后一行的数据，准备计算调整的方差
## 注意，保留的日期系exr_f的日期
mkt_ajsta0 = pd.merge(mkt_wol, mkt_wof, on = 'order', how = 'left')
mkt_ajsta0 = mkt_ajsta0[[ 'year', 'month', 'date', 'exr_f', 'exr_l']]

In [12]:
## 合并均值数据
mkt_ajsta = pd.merge(mkt_ajsta0, mkt_sta[['year', 'month', 'mean']], on = ['year', 'month'], how = 'left')
mkt_ajsta['diff1'] = mkt_ajsta['exr_f'] - mkt_ajsta['mean']
mkt_ajsta['diff2'] = mkt_ajsta['exr_l'] - mkt_ajsta['mean']
mkt_ajsta['product'] = mkt_ajsta['diff1'] * mkt_ajsta['diff2']
mkt_ajsta = mkt_ajsta.groupby([mkt_ajsta['year'], mkt_ajsta['month']])['product'].agg([('prdct', 'sum'),])
mkt_ajsta = mkt_ajsta.reset_index()

In [13]:
## 合并调整后的数据和原统计指标值
mkt_ajs = pd.merge(mkt_sta, mkt_ajsta, on = ['year', 'month'], how = 'left' )
mkt_ajs['var'] = mkt_ajs['math_var'] * mkt_ajs['num'] + 2* mkt_ajs['prdct']
mkt_ajs['adj_var'] = mkt_ajs['var']/mkt_ajs['num']

In [14]:
mkt_prdct = mkt_ajs[['year', 'month', 'num', 'mean', 'var', 'adj_var', 'math_skew', 'math_kurt']]

In [15]:
mkt_prdct

,year,month,num,mean,var,adj_var,math_skew,math_kurt
0,2007,1,20,-0.006018,0.019452,0.000973,-0.838896,0.203171
1,2007,2,15,-0.016386,0.003652,0.000243,-2.066747,3.529964
2,2007,3,22,-0.010252,0.003059,0.000139,-0.664393,-0.448320
3,2007,4,21,-0.011775,0.006224,0.000296,-1.998115,4.018785
4,2007,5,18,-0.012575,0.011132,0.000618,-1.810055,2.828659
...,...,...,...,...,...,...,...,...
140,2018,9,19,-0.024065,0.001182,0.000062,0.240583,-0.934328
141,2018,10,18,-0.028148,0.008161,0.000453,-0.274263,0.232362
142,2018,11,22,-0.022886,0.004063,0.000185,-0.373812,0.710656
143,2018,12,20,-0.026019,0.002795,0.000140,0.538603,0.485477


In [18]:
mkt_prdct.to_csv("Weight and Ave/mkt_prdct.csv", index = False)